<a href="https://colab.research.google.com/github/bigbadcyborg/computer-vision-project/blob/dev/computer_vision_11_11(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator
import cv2
from google.colab.patches import cv2_imshow
from google.colab import files # Import files module
import numpy as np
import time # Import the time module
import io # Import io for file handling

# Load model
model = YOLO('yolov8m.pt')  # You can use yolov8n.pt for faster results

# Set classes to detect (person=0, car=2, cat=15, dog=16)
targetClasses = [0, 2, 15, 16]

# Initialize traffic light state and timers
current_light_state = "Green"
state_start_time = time.time()
green_light_duration = 10 # seconds
yellow_light_duration = 3 # seconds
red_light_duration = 15 # seconds

# Initialize variable to store car bounding boxes from the previous frame
# Format: [(bbox_xyxy, centroid_x, centroid_y), ...]
previous_frame_car_boxes = []

# Define a movement threshold in pixels
movement_threshold = 10  # Adjust as needed

# --- File Upload Code ---
print("Please upload your video file.")
uploaded = files.upload()

if not uploaded:
    print("No file uploaded. Exiting.")
    exit()

# Assuming only one file is uploaded
video_filename = next(iter(uploaded))
video_content = uploaded[video_filename]

# Save the uploaded video content to a temporary file
# This is necessary because cv2.VideoCapture often expects a file path
# and can't directly read from in-memory bytes reliably.
tmp_video_path = f"/tmp/{video_filename}"
with open(tmp_video_path, "wb") as f:
    f.write(video_content)

cap = cv2.VideoCapture(tmp_video_path)

# Check if opened
if not cap.isOpened():
    print(f"Error opening video file: {tmp_video_path}")
    exit()

# Get video properties for VideoWriter
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Define the codec and create VideoWriter object
output_video_path = 'detection-output.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec for .mp4 files
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

frame_count = 0
# NEW: Counter for iterations with no detections while in Red light state
no_detection_red_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break # End of video or error reading frame

    frame_count += 1

    if frame_count % 5 == 0: # Process every 5th frame
        print(f"Processing frame {frame_count}...")
        results = model(frame)[0]  # Predict on current frame

        annotator = Annotator(frame.copy()) # Use a copy to avoid modifying original frame for multiple annotations

        detected_objects_in_frame = []
        is_moving_car_detected = False
        is_person_cat_or_dog_detected = False # Initialize person/pet detection flag
        current_frame_car_boxes = []

        for box in results.boxes:
            cls = int(box.cls)
            if cls in targetClasses:
                b = box.xyxy[0].cpu().numpy() # Get box coordinates and convert to numpy array
                label = model.names[cls]
                annotator.box_label(b, label)
                detected_objects_in_frame.append(label)

                if cls == 2: # If it's a car
                    # Calculate centroid of the current car's bounding box
                    c_x = (b[0] + b[2]) / 2
                    c_y = (b[1] + b[3]) / 2
                    current_car_centroid = (c_x, c_y)

                    # Check for movement from previous frame's cars
                    for prev_bbox, prev_centroid_x, prev_centroid_y in previous_frame_car_boxes:
                        distance = np.sqrt((current_car_centroid[0] - prev_centroid_x)**2 +
                                           (current_car_centroid[1] - prev_centroid_y)**2)
                        if distance > movement_threshold:
                            is_moving_car_detected = True
                            break # Found a moving car, no need to check other previous cars for this current car

                    current_frame_car_boxes.append((b, c_x, c_y))

                # Check for person, cat or dog detection
                if cls == 0 or cls == 15 or cls == 16: # 0 is person, 15 is cat, 16 is dog
                    is_person_cat_or_dog_detected = True

        # Update previous frame car boxes for the next iteration
        previous_frame_car_boxes = current_frame_car_boxes

        # --- Traffic Light State Machine Logic ---
        elapsed_time = time.time() - state_start_time
        trigger_message = ""

        # Update no_detection_red_count based on current state and *person/cat/dog* detections
        if current_light_state == "Red":
            if is_person_cat_or_dog_detected: # Only reset if person/cat/dog is detected
                no_detection_red_count = 0
            else:
                no_detection_red_count += 1 # Increment if no person/cat/dog
        else:
            no_detection_red_count = 0 # Reset when not in Red state

        if current_light_state == "Green":
            # Transition to Yellow only if green_light_duration has passed.
            # Detections will set trigger_message if present at the time of transition.
            if elapsed_time >= green_light_duration:
                current_light_state = "Yellow"
                state_start_time = time.time()
                # Set trigger message if detections were present when the light turned yellow
                if is_moving_car_detected and is_person_cat_or_dog_detected:
                    trigger_message = " - Moving Car & Person/Pet Detected"
                elif is_moving_car_detected:
                    trigger_message = " - Moving Car Detected"
                elif is_person_cat_or_dog_detected:
                    trigger_message = " - Person/Pet Detected"
        elif current_light_state == "Yellow":
            if elapsed_time >= yellow_light_duration:
                current_light_state = "Red"
                state_start_time = time.time()
        elif current_light_state == "Red":
            # Only turn Green if red_light_duration has passed AND there's no person/cat/dog for at least 2 consecutive processed frames
            # This allows turning green even with moving cars, as long as the pedestrian/animal path is clear.
            if elapsed_time >= red_light_duration and no_detection_red_count >= 2:
                current_light_state = "Green"
                state_start_time = time.time()

        # Assign the state from the state machine to display
        display_light_state = current_light_state + " Light"

        # Annotate streetlight state on the frame
        annotatedFrame = annotator.result()

        # Define text properties
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1
        font_thickness = 2
        text_color_green = (0, 255, 0) # Green
        text_color_yellow = (0, 255, 255) # Yellow
        text_color_red = (0, 0, 255) # Red (new color)
        background_color = (0, 0, 0) # Black

        # 1. Update text_to_display to only show the display_light_state
        text_to_display_label = display_light_state
        (text_width, text_height), baseline = cv2.getTextSize(text_to_display_label, font, font_scale, font_thickness)

        # Position the text at the top-left corner with some padding
        padding = 10
        text_position = (padding, padding + text_height)

        # Draw a filled rectangle as background for the text
        cv2.rectangle(annotatedFrame, (padding, padding), (padding + text_width + 10, padding + text_height + baseline + 10), background_color, -1)

        # Put the text on the image with appropriate color
        if current_light_state == "Green":
            cv2.putText(annotatedFrame, text_to_display_label, text_position, font, font_scale, text_color_green, font_thickness, cv2.LINE_AA)
            light_circle_color = text_color_green
        elif current_light_state == "Yellow":
            cv2.putText(annotatedFrame, text_to_display_label, text_position, font, font_scale, text_color_yellow, font_thickness, cv2.LINE_AA)
            light_circle_color = text_color_yellow
        else: # Red Light
            cv2.putText(annotatedFrame, text_to_display_label, text_position, font, font_scale, text_color_red, font_thickness, cv2.LINE_AA)
            light_circle_color = text_color_red

        # 2. and 3. Draw a circle representing the traffic light color
        # Get frame dimensions for positioning
        h, w, _ = annotatedFrame.shape
        circle_radius = 20
        circle_center = (w - padding - circle_radius, padding + circle_radius) # Top-right corner

        # 4. Use cv2.circle() to draw the circle
        cv2.circle(annotatedFrame, circle_center, circle_radius, light_circle_color, -1) # -1 for filled circle
        cv2.circle(annotatedFrame, circle_center, circle_radius, (255,255,255), 2) # White outline

        if detected_objects_in_frame or current_light_state != "Green": # Display if objects detected or light is not green
            print(f"Detected objects in frame {frame_count}: {', '.join(set(detected_objects_in_frame))}")
            if is_moving_car_detected:
                print(f"  -> Moving car detected in frame {frame_count}!")
            if is_person_cat_or_dog_detected:
                print(f"  -> Person/Cat/Dog detected in frame {frame_count}!")
            print(f"  -> Streetlight State: {display_light_state}{trigger_message}") # Keep full message for console output
            cv2_imshow(annotatedFrame)

        # Write the annotated frame to the output video
        out.write(annotatedFrame)

cap.release()
out.release() # Release the VideoWriter
print("Video processing complete.")
print(f"Output video saved to {output_video_path}")